In [1]:
# 08_robustness_and_baselines.ipynb
# Robustness study: (A) incumbent & glassbox comparators, (B) subgroup fairness,
# (C) external-portfolio replication. Self-contained; reuses saved artifacts
# and the shared engine in common.py. Grayscale, dpi 600, PNG+PDF, no captions.

import os, warnings, pickle, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

import config as CFG
import common as C
import data_loaders as D
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

C.set_grayscale_style()
RS = CFG.RANDOM_STATE
PROC = os.path.join("..", "data", "processed")
NB  = CFG.N_BINS
LAM = CFG.LAMBDA_MIX

def _metrics_row(name, y, p):
    bm = C.brier_murphy(y, p)
    return {"model": name, "AUC": C.auc(y, p), "ECE": C.ece(y, p),
            "reliability": bm["reliability"], "refinement": bm["refinement"]}

# ===========================================================================
# PART A — Incumbent (logistic WoE) and glassbox (EBM) comparators
#          on the SAME saved evaluation split.
# ===========================================================================
print("=" * 68, "\n[A] Baseline comparators on the held-out evaluation split\n" + "=" * 68)

X = pd.read_pickle(os.path.join(PROC, "X_all.pkl"))
y = np.load(os.path.join(PROC, "y_all.npy"))
with open(os.path.join(PROC, "splits.pkl"), "rb") as f: SP = pickle.load(f)
tp = np.load(os.path.join(PROC, "teacher_probs.npz"))
with open(os.path.join(PROC, "distilled_models.pkl"), "rb") as f: DM = pickle.load(f)
with open(os.path.join(PROC, "meta.json")) as f: META = json.load(f)
feats = META["features"]
itr, iev = SP["train"], SP["eval"]
Xtr, ytr = X.iloc[itr].reset_index(drop=True), y[itr]
Xev, yev = X.iloc[iev].reset_index(drop=True), y[iev]

rows = []
# teacher and the two saved scorecards (continuous probabilities)
rows.append(_metrics_row("TabPFN v2 teacher", yev, tp["eval"]))
for key, label in [("Baseline", "Scorecard (hard label)"), ("Full", "Scorecard (distilled)")]:
    sc, bn = DM[key]["scorecard"], DM[key]["binning"]
    p = sc.predict_proba(C.apply_binning(Xev, bn))
    rows.append(_metrics_row(label, yev, p))

# --- Incumbent: logistic regression on weight-of-evidence, fit on train ---
def _woe_fit_apply(xtr, ytr_, xev):
    edges = C.quantile_edges(xtr, NB)
    _, wmap = C.woe_transform(xtr, ytr_, edges)
    def _ap(x):
        idx = np.digitize(np.where(np.isnan(x), -np.inf, x), edges, right=False)
        return np.array([wmap.get(int(b), 0.0) for b in idx])
    return _ap(xtr), _ap(xev)

Wtr = np.column_stack([_woe_fit_apply(Xtr[f].to_numpy(float), ytr, Xev[f].to_numpy(float))[0] for f in feats])
Wev = np.column_stack([_woe_fit_apply(Xtr[f].to_numpy(float), ytr, Xev[f].to_numpy(float))[1] for f in feats])
lr = LogisticRegression(max_iter=2000).fit(Wtr, ytr)
rows.append(_metrics_row("Logistic WoE scorecard", yev, lr.predict_proba(Wev)[:, 1]))

# --- Glassbox: Explainable Boosting Machine (optional dependency) ---
try:
    from interpret.glassbox import ExplainableBoostingClassifier
    ebm = ExplainableBoostingClassifier(random_state=RS)
    ebm.fit(Xtr[feats], ytr)
    rows.append(_metrics_row("EBM (GA2M glassbox)", yev, ebm.predict_proba(Xev[feats])[:, 1]))
except Exception as e:
    print("  [skip] EBM unavailable (pip install interpret):", type(e).__name__)

tblA = pd.DataFrame(rows)[["model", "AUC", "ECE", "reliability", "refinement"]]
print(tblA.to_string(index=False))
C.save_table(tblA, "08_baseline_comparators")

# grayscale AUC comparison with the teacher as a reference line
order = tblA.sort_values("AUC")
fig, ax = plt.subplots(figsize=(7.2, 4.05))            # 16:9
ax.barh(order["model"], order["AUC"], color="0.55", edgecolor="black")
tea = tblA.loc[tblA["model"].str.contains("teacher"), "AUC"].iloc[0]
ax.axvline(tea, color="black", ls="--", lw=1.2)
ax.set_xlabel("evaluation AUC"); ax.set_xlim(min(order["AUC"]) - 0.01, max(order["AUC"]) + 0.005)
C.save_fig(fig, "08_baseline_auc"); plt.close(fig)

# ===========================================================================
# PART B — Subgroup fairness on the protected attribute (gender).
#          Reconstructs the aligned attribute from the raw file using the
#          same deterministic subsample; runs only for Home Credit.
# ===========================================================================
print("\n" + "=" * 68, "\n[B] Subgroup fairness audit (gender)\n" + "=" * 68)
raw_path = os.path.join("..", "data", "raw", getattr(CFG, "HOME_CREDIT_FILE", ""))
sub_n = getattr(CFG, "SUBSAMPLE_N", {}).get("home_credit", None)
if getattr(CFG, "DATASET", "") == "home_credit" and os.path.exists(raw_path):
    raw = pd.read_csv(raw_path, usecols=lambda c: c in ("CODE_GENDER",))
    idx = np.random.default_rng(RS).choice(len(raw), int(sub_n), replace=False)  # mirrors _subsample
    g_all = raw["CODE_GENDER"].to_numpy()[idx]
    assert len(g_all) == len(X), "subsample alignment mismatch; check SUBSAMPLE_N / RANDOM_STATE"
    g_ev = g_all[iev]
    keep = np.isin(g_ev, ["M", "F"])
    g_ev, y_f = g_ev[keep], yev[keep]

    def _scores(name):
        if name == "teacher": return tp["eval"][keep]
        k = "Baseline" if "hard" in name else "Full"
        p = DM[k]["scorecard"].predict_proba(C.apply_binning(Xev, DM[k]["binning"]))
        return p[keep]

    TAU = 0.70                                    # common approval rate
    frows = []
    for name, disp in [("teacher", "Teacher"), ("hard", "Scorecard (hard)"), ("dist", "Scorecard (distilled)")]:
        p = _scores(name)
        thr = np.quantile(p, TAU)                 # approve lowest-risk TAU share
        appr = p <= thr
        ar = {gg: appr[g_ev == gg].mean() for gg in ["M", "F"]}
        air = min(ar.values()) / max(ar.values())                       # adverse impact ratio
        good = (y_f == 0)                                                # would repay
        eo = {gg: appr[(g_ev == gg) & good].mean() for gg in ["M", "F"]}
        eod = eo["F"] - eo["M"]                                          # equal-opportunity diff
        dr = {gg: y_f[appr & (g_ev == gg)].mean() for gg in ["M", "F"]}  # approved default rate
        frows.append({"model": disp, "approval_M": ar["M"], "approval_F": ar["F"],
                      "adverse_impact_ratio": air, "equal_opp_diff": eod,
                      "def_rate_M": dr["M"], "def_rate_F": dr["F"]})
    tblB = pd.DataFrame(frows)
    print(tblB.to_string(index=False))
    C.save_table(tblB, "08_fairness_gender")

    fig, ax = plt.subplots(figsize=(7.2, 4.05))
    xpos = np.arange(len(tblB)); w = 0.38
    ax.bar(xpos - w / 2, tblB["approval_M"], w, color="0.35", edgecolor="black", label="M")
    ax.bar(xpos + w / 2, tblB["approval_F"], w, color="0.75", edgecolor="black", label="F")
    ax.set_xticks(xpos); ax.set_xticklabels(tblB["model"]); ax.set_ylabel("approval rate")
    ax.legend(frameon=False)
    C.save_fig(fig, "08_fairness_gender"); plt.close(fig)
else:
    print("  [skip] needs DATASET='home_credit' and the raw CSV present.")

# ===========================================================================
# PART C — External-portfolio replication of the core claims with a real
#          TabPFN teacher (teacher quality, 2x2 ablation, calibration transfer).
#          Guarded: requires internet (OpenML) and the tabpfn package.
# ===========================================================================
print("\n" + "=" * 68, "\n[C] External-portfolio replication\n" + "=" * 68)
try:
    from sklearn.datasets import fetch_openml
    from tabpfn import TabPFNClassifier

    # second real credit portfolio (Taiwan default; falls back to German credit)
    try:
        ds = fetch_openml("default-of-credit-card-clients", version=1, as_frame=True)
        Xe = ds.data.apply(pd.to_numeric, errors="coerce")
        ye = (ds.target.astype(str).str.strip().isin(["1", "yes", "Y"])).astype(int).to_numpy()
        ext_name = "Taiwan default"
    except Exception:
        ds = fetch_openml("credit-g", version=1, as_frame=True)
        Xe = ds.data.apply(pd.to_numeric, errors="coerce")
        Xe = Xe.loc[:, Xe.notna().any()]
        ye = (ds.target.astype(str) == "bad").astype(int).to_numpy()
        ext_name = "German credit"
    Xe = Xe.select_dtypes("number").reset_index(drop=True)
    print(f"  dataset: {ext_name}  shape={Xe.shape}  default_rate={ye.mean():.3f}")

    # class-ratio-preserving subsample + triple split
    n = min(len(Xe), 16000)
    keep, _ = train_test_split(np.arange(len(Xe)), train_size=n, stratify=ye, random_state=RS)
    Xe, ye = Xe.iloc[keep].reset_index(drop=True), ye[keep]
    itr2, itmp = train_test_split(np.arange(n), train_size=0.6, stratify=ye, random_state=RS)
    ies2, iev2 = train_test_split(itmp, train_size=0.5, stratify=ye[itmp], random_state=RS)
    fe = list(Xe.columns)
    Xtr2, Xes2, Xev2 = (Xe.iloc[i].reset_index(drop=True) for i in (itr2, ies2, iev2))
    ytr2, yes2, yev2 = ye[itr2], ye[ies2], ye[iev2]

    # teacher: prevalence-preserving context <= cap, batched prediction
    dev = C.torch_device(); med = Xtr2.median(numeric_only=True)
    mp = getattr(CFG, "TABPFN_MODEL_PATH", None)
    clf = TabPFNClassifier(device=dev, random_state=RS, **({"model_path": mp} if mp else {}))
    cap = getattr(CFG, "TEACHER_ROW_CAP", 10000)
    if len(Xtr2) > cap:
        take, _ = train_test_split(np.arange(len(Xtr2)), train_size=cap, stratify=ytr2, random_state=RS)
        clf.fit(Xtr2.iloc[take].fillna(med), ytr2[take])
    else:
        clf.fit(Xtr2.fillna(med), ytr2)
    bs = getattr(CFG, "TEACHER_PREDICT_BATCH", 1000)
    def _tea(Xin):
        Xf = Xin.fillna(med); out = []
        for i in range(0, len(Xf), bs):
            out.append(clf.predict_proba(Xf.iloc[i:i + bs])[:, 1])
        return np.concatenate(out)
    pT_tr, pT_ev = _tea(Xtr2), _tea(Xev2)
    tea_auc, tea_cal = C.auc(yev2, pT_ev), C.brier_murphy(yev2, pT_ev)["reliability"]

    # 2x2 ablation with the shared engine
    bn_std = C.fit_binning(Xtr2, ytr2,  fe, NB, "standard")
    bn_ind = C.fit_binning(Xtr2, pT_tr, fe, NB, "TFM_induced")
    Btr = {"std": C.apply_binning(Xtr2, bn_std), "ind": C.apply_binning(Xtr2, bn_ind)}
    Bes = {"std": C.apply_binning(Xes2, bn_std), "ind": C.apply_binning(Xes2, bn_ind)}
    Bev = {"std": C.apply_binning(Xev2, bn_std), "ind": C.apply_binning(Xev2, bn_ind)}
    tgt_hard, tgt_dist = ytr2.astype(float), (1 - LAM) * pT_tr + LAM * ytr2
    cond = {"Baseline": ("std", tgt_hard), "H2-only": ("ind", tgt_hard),
            "H1-only": ("std", tgt_dist), "Full": ("ind", tgt_dist)}
    arows, aucs = [], {}
    for name, (st, tg) in cond.items():
        sc = C.fit_stump_scorecard(Btr[st], tg, Bes[st], yes2)
        a = C.auc(yev2, sc.predict_proba(Bev[st])); aucs[name] = a
        arows.append({"condition": name, "AUC": a,
                      "reliability": C.brier_murphy(yev2, sc.predict_proba(Bev[st]))["reliability"]})
    gap = (aucs["Full"] - aucs["Baseline"]) / max(tea_auc - aucs["Baseline"], 1e-9)
    arows.append({"condition": "Teacher", "AUC": tea_auc, "reliability": tea_cal})
    tblC = pd.DataFrame(arows)
    tblC.attrs["gap_recovered"] = gap
    print(f"  teacher AUC={tea_auc:.3f} reliab={tea_cal:.1e} | gap recovered={gap:.0%}")
    print(tblC.to_string(index=False))
    C.save_table(tblC, f"08_external_ablation")

    # calibration transfer (standard structure both; score isolated)
    p_td = C.fit_stump_scorecard(Btr["std"], tgt_dist, Bes["std"], yes2).predict_proba(Bev["std"])
    p_mis = C.sigmoid(C.logit(pT_tr) / 0.5)                        # rank-preserving miscalibration
    p_mc = C.fit_stump_scorecard(Btr["std"], (1 - LAM) * p_mis + LAM * ytr2,
                                 Bes["std"], yes2).predict_proba(Bev["std"])
    dl = C.delong_test(yev2, p_td, p_mc)
    print(f"  gate-beta: DeLong p(equal AUC)={dl['p_value']:.2f} | "
          f"reliab teacher-distilled={C.brier_murphy(yev2,p_td)['reliability']:.1e} "
          f"vs miscalibrated={C.brier_murphy(yev2,p_mc)['reliability']:.1e}")
    tblT = pd.DataFrame([
        {"variant": "teacher-distilled", "AUC": C.auc(yev2, p_td),
         "reliability": C.brier_murphy(yev2, p_td)["reliability"]},
        {"variant": "rank-miscalibrated", "AUC": C.auc(yev2, p_mc),
         "reliability": C.brier_murphy(yev2, p_mc)["reliability"]},
    ])
    tblT.attrs["delong_p"] = dl["p_value"]
    C.save_table(tblT, "08_external_calibration_transfer")

    # grayscale ablation figure (external)
    fig, ax = plt.subplots(figsize=(7.2, 4.05))
    o = ["Baseline", "H2-only", "H1-only", "Full"]
    ax.plot(o, [aucs[k] for k in o], "-o", color="black")
    ax.axhline(tea_auc, color="0.4", ls="--", lw=1.2)
    ax.set_ylabel("evaluation AUC")
    C.save_fig(fig, "08_external_ablation"); plt.close(fig)
except Exception as e:
    print("  [skip] external replication not run:", type(e).__name__, str(e)[:120])

print("\n[done] Part A/B always produce outputs; C runs when TabPFN + internet are available.")

[A] Baseline comparators on the held-out evaluation split
  [skip] EBM unavailable (pip install interpret): ModuleNotFoundError
                 model      AUC      ECE  reliability  refinement
     TabPFN v2 teacher 0.736068 0.014199     0.000558    0.004668
Scorecard (hard label) 0.733037 0.018380     0.001039    0.005639
 Scorecard (distilled) 0.732785 0.014197     0.000617    0.004881
Logistic WoE scorecard 0.727887 0.016943     0.000397    0.004632
[saved] 08_baseline_comparators.csv
[saved] 08_baseline_auc.png  &  08_baseline_auc.pdf

[B] Subgroup fairness audit (gender)
                model  approval_M  approval_F  adverse_impact_ratio  equal_opp_diff  def_rate_M  def_rate_F
              Teacher    0.662313    0.718985              0.921178        0.053537    0.056338    0.039869
     Scorecard (hard)    0.651119    0.725094              0.897979        0.070466    0.054441    0.039533
Scorecard (distilled)    0.667910    0.716165              0.932620        0.048950    0.062

In [3]:
# 08_partA_comparators.ipynb  (rerun: now includes EBM glassbox)
import os, pickle, json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import config as CFG, common as C
from sklearn.linear_model import LogisticRegression
C.set_grayscale_style()
PROC = os.path.join("..", "data", "processed"); NB = CFG.N_BINS

X = pd.read_pickle(os.path.join(PROC, "X_all.pkl")); y = np.load(os.path.join(PROC, "y_all.npy"))
with open(os.path.join(PROC, "splits.pkl"), "rb") as f: SP = pickle.load(f)
tp = np.load(os.path.join(PROC, "teacher_probs.npz"))
with open(os.path.join(PROC, "distilled_models.pkl"), "rb") as f: DM = pickle.load(f)
feats = json.load(open(os.path.join(PROC, "meta.json")))["features"]
Xtr, ytr = X.iloc[SP["train"]].reset_index(drop=True), y[SP["train"]]
Xev, yev = X.iloc[SP["eval"]].reset_index(drop=True), y[SP["eval"]]

def _row(name, yv, p):
    bm = C.brier_murphy(yv, p)
    return {"model": name, "AUC": C.auc(yv, p), "ECE": C.ece(yv, p),
            "reliability": bm["reliability"], "refinement": bm["refinement"]}

rows = [_row("TabPFN v2 teacher", yev, tp["eval"])]
for k, lab in [("Baseline", "Scorecard (hard label)"), ("Full", "Scorecard (distilled)")]:
    p = DM[k]["scorecard"].predict_proba(C.apply_binning(Xev, DM[k]["binning"]))
    rows.append(_row(lab, yev, p))

# incumbent: logistic weight-of-evidence, fit on train
def _woe(xtr, ytr_, xev):
    e = C.quantile_edges(xtr, NB); _, wm = C.woe_transform(xtr, ytr_, e)
    ap = lambda x: np.array([wm.get(int(b), 0.0) for b in
                             np.digitize(np.where(np.isnan(x), -np.inf, x), e, right=False)])
    return ap(xtr), ap(xev)
Wtr = np.column_stack([_woe(Xtr[f].to_numpy(float), ytr, Xev[f].to_numpy(float))[0] for f in feats])
Wev = np.column_stack([_woe(Xtr[f].to_numpy(float), ytr, Xev[f].to_numpy(float))[1] for f in feats])
lr = LogisticRegression(max_iter=2000).fit(Wtr, ytr)
rows.append(_row("Logistic WoE scorecard", yev, lr.predict_proba(Wev)[:, 1]))

# glassbox: Explainable Boosting Machine
from interpret.glassbox import ExplainableBoostingClassifier
ebm = ExplainableBoostingClassifier(random_state=CFG.RANDOM_STATE).fit(Xtr[feats], ytr)
rows.append(_row("EBM (GA2M glassbox)", yev, ebm.predict_proba(Xev[feats])[:, 1]))

tblA = pd.DataFrame(rows)[["model", "AUC", "ECE", "reliability", "refinement"]]
print(tblA.to_string(index=False)); C.save_table(tblA, "08_baseline_comparators")

order = tblA.sort_values("AUC")
fig, ax = plt.subplots(figsize=(7.2, 4.05))
ax.barh(order["model"], order["AUC"], color="0.55", edgecolor="black")
ax.axvline(tblA.loc[tblA["model"].str.contains("teacher"), "AUC"].iloc[0], color="black", ls="--", lw=1.2)
ax.set_xlabel("evaluation AUC"); ax.set_xlim(min(order["AUC"]) - 0.01, max(order["AUC"]) + 0.005)
C.save_fig(fig, "08_baseline_auc"); plt.close(fig)

                 model      AUC      ECE  reliability  refinement
     TabPFN v2 teacher 0.736068 0.014199     0.000558    0.004668
Scorecard (hard label) 0.733037 0.018380     0.001039    0.005639
 Scorecard (distilled) 0.732785 0.014197     0.000617    0.004881
Logistic WoE scorecard 0.727887 0.016943     0.000397    0.004632
   EBM (GA2M glassbox) 0.729360 0.013568     0.000289    0.005101
[saved] 08_baseline_comparators.csv
[saved] 08_baseline_auc.png  &  08_baseline_auc.pdf


In [4]:
# regen: teacher reliability diagram zoomed to the populated PD range
import os, pickle, numpy as np, matplotlib.pyplot as plt
import config as CFG, common as C
C.set_grayscale_style()
PROC = os.path.join("..", "data", "processed")
y = np.load(os.path.join(PROC, "y_all.npy"))
with open(os.path.join(PROC, "splits.pkl"), "rb") as f: SP = pickle.load(f)
p = np.load(os.path.join(PROC, "teacher_probs.npz"))["eval"]; yev = y[SP["eval"]]

q = np.unique(np.quantile(p, np.linspace(0, 1, 11)))     # decile bins on the forecast
ids = np.digitize(p, q[1:-1])
mp = np.array([p[ids == b].mean() for b in np.unique(ids)])
of = np.array([yev[ids == b].mean() for b in np.unique(ids)])
hi = float(max(mp.max(), of.max()) * 1.08)               # zoom to occupied range

fig, ax = plt.subplots(figsize=(4.6, 4.6))
ax.plot([0, hi], [0, hi], ls="--", color="0.6", lw=1.1)  # ideal calibration
ax.plot(mp, of, "-o", color="black")
ax2 = ax.twinx()                                          # forecast distribution (context)
ax2.hist(p, bins=30, range=(0, hi), color="0.85", edgecolor="none", zorder=0)
ax2.set_yticks([]); ax2.set_zorder(0); ax.set_zorder(1); ax.patch.set_visible(False)
ax.set_xlim(0, hi); ax.set_ylim(0, hi)
ax.set_xlabel("mean predicted probability"); ax.set_ylabel("observed default frequency")
C.save_fig(fig, "01_teacher_reliability"); plt.close(fig)

[saved] 01_teacher_reliability.png  &  01_teacher_reliability.pdf
